# LabLens - SFT with Unsloth (fixed)
Settings: **GPU T4 x2**, **Internet On**, dataset attached.
**Run cells top to bottom.** If you change the model, do *Run -> Restart & clear outputs* first.

In [ ]:
!pip install -q unsloth

In [ ]:
# 1. Use ONE GPU and locate the data. This must run BEFORE importing unsloth / torch.
import os, glob, json, random, re
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

files = glob.glob("/kaggle/input/**/extract_train.jsonl", recursive=True)
assert files, "Dataset not found - attach it with '+ Add Input'"
DATA = os.path.dirname(files[0])
print("Data folder:", DATA)

def load(name):
    with open(os.path.join(DATA, name), encoding="utf-8") as f:
        return [json.loads(line) for line in f]

TASK = "extract"          # "extract" (report -> JSON) or "drift" (history -> trends)
train = load(f"{TASK}_train.jsonl")
test  = load(f"{TASK}_test.jsonl")
print(TASK, "train:", len(train), "test:", len(test))

In [ ]:
# 2. Load a SMALL model (1.5B) - fits easily on one T4 and trains in fp16
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = False,       # 1.5B is small enough for 16-bit LoRA
    dtype = None,               # auto: fp16 on T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 3. Turn each chat (system, user, assistant) into ONE 'text' string - Unsloth needs this
from datasets import Dataset

def to_text(rows):
    return Dataset.from_list([
        {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows
    ])

random.seed(42)
dataset = to_text(random.sample(train, min(1500, len(train))))
print(dataset)
print(dataset[0]["text"][:700])

In [ ]:
# 4. Trainer + compute loss only on the assistant's answer
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,     # effective batch 8
        num_train_epochs = 1,                # 1500 rows -> ~188 steps
        warmup_steps = 10,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        output_dir = "outputs",
        save_strategy = "no",
        seed = 3407,
        dataset_num_proc = 1,
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Check: this should print ONLY the JSON answer
labels = trainer.train_dataset[0]["labels"]
print(tokenizer.decode([t for t in labels if t != -100])[:500])

In [ ]:
# 5. Train (~30-60 min on a T4). Watch the loss go down in the log.
stats = trainer.train()

## 6. Evaluate on the held-out test set
For `extract`, the test set is the unseen Kaveri layout.

In [ ]:
def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def score_extract(pred, gt):
    if not isinstance(pred, dict) or not isinstance(pred.get("results"), list):
        return 0.0
    want = {r["test"]: r for r in gt["results"]}
    got = {r["test"]: r for r in pred["results"] if isinstance(r, dict) and "test" in r}
    s = 0.0
    for k, g in want.items():
        r = got.get(k)
        if not r:
            continue
        try:
            ok_v = abs(float(r.get("value")) - g["value"]) <= max(0.01 * abs(g["value"]), 0.011)
        except (TypeError, ValueError):
            ok_v = False
        s += 0.6 * ok_v + 0.2 * (r.get("unit") == g["unit"]) + 0.2 * (r.get("flag") == g["flag"])
    invented = sum(1 for k in got if k not in want) / max(len(got), 1)
    return max(0.0, s / len(want) - 0.5 * invented)

def score_drift(pred, gt):
    if not isinstance(pred, dict) or not isinstance(pred.get("trends"), list):
        return 0.0
    got = {t.get("test"): t for t in pred["trends"] if isinstance(t, dict)}
    s = 0.0
    for g in gt["trends"]:
        p = got.get(g["test"], {})
        s += 0.5 * (p.get("direction") == g["direction"]) + 0.5 * (p.get("in_range_drift") == g["in_range_drift"])
    return s / len(gt["trends"])

score_fn = score_extract if TASK == "extract" else score_drift

FastLanguageModel.for_inference(model)
random.seed(0)
EVAL = random.sample(test, 20)
scores = []
for i, row in enumerate(EVAL):
    msgs = row["messages"]
    prompt = tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=1200, do_sample=False, use_cache=True)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    scores.append(score_fn(parse_json(text), json.loads(msgs[2]["content"])))
    if i == 0:
        print("MODEL OUTPUT:\n", text[:800])
print(f"\nAverage score on {len(EVAL)} test examples: {sum(scores)/len(scores):.3f}")

In [ ]:
# 7. Save the LoRA adapter (download it from the Output tab on the right)
model.save_pretrained("/kaggle/working/lablens-sft")
tokenizer.save_pretrained("/kaggle/working/lablens-sft")